# Install dependencies from uv and setup reloading of imports.

In [13]:
!uv sync
%load_ext autoreload
%autoreload 2

Resolved 121 packages in 5ms
Checked 118 packages in 42ms
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create the agents.

In [88]:
from google.adk.agents import Agent

import config
import instructions
import tools

weather_agent = Agent(
    name="weather_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("weather-agent-instructions"),
    tools=[tools.get_weather, tools.get_lat_lon]
)

# Perform tests on local agent.

In [89]:
import agent_tester

tester = agent_tester.AgentTester(weather_agent)

print("================ New York ===========================")
await tester.run_prompt("What is the weather for New York City, New York")
print("================ Reston =============================")
await tester.run_prompt("What is the weather for Reston, VA")
print("================ Los Angeles ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA")

================ New York ===========================
📍 Weather for New York City, New York 📅 Friday, July 19, 2024                 

🌤️ Conditions:     Sunny then Chance Showers And Thunderstorms 🌡️ Temperature:  
86°F  |  High: 86°F  Low: 77°F 💧 Humidity:       N/A 💨 Wind:           SW 3 to
8 mph 🌧️ Precipitation:  Chance of showers and thunderstorms (40% during the    
day, 60% tonight) 👁️ Visibility:     N/A 🌅 Sunrise:        N/A 🌇 Sunset:      
N/A                                                                             
================ Reston =============================
📍 Weather for Reston, VA 📅 Friday, July 26, 2024                              

🌤️ Conditions:     Partly Sunny then Chance Showers And Thunderstorms 🌡️        
Temperature:    90°F  |  High: 90°F  Low: 71°F 💧 Humidity:       Not available 
💨 Wind:           3 to 8 mph SW 🌧️ Precipitation:  A chance of showers and     
thunderstorms after 2pm. Chance of precipitation is 70%. New rainfall amounts   
between a

# Deploy the agent.

In [90]:
import vertexai
from vertexai import types
from vertexai import agent_engines

import config

client = vertexai.Client(project=config.PROJECT_ID, location=config.LOCATION)

app = agent_engines.AdkApp(agent=weather_agent, app_name="weather_agent")

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "display_name": "weather_agent",
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
        "staging_bucket": config.STAGING_BUCKET,
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "extra_packages": [
            "callbacks.py",
            "config.py",
            "tools.py",
            "instructions.py",
            "./resources"
        ],
        "env_vars": {
            "GOOGLE_MAPS_KEY": config.GOOGLE_MAPS_KEY,
            "PROJECT_ID": config.PROJECT_ID,
            "STAGING_BUCKET": config.STAGING_BUCKET,
            "LOCATION": config.LOCATION
        }
    }
)

The following requirements are missing: {'cloudpickle', 'pydantic'}


# Perform tests on the remote agent.

In [92]:
import agent_tester

print("================ New York ===========================")
await agent_tester.run_remote_agent_prompt(remote_agent, "What is the weather for New York City, New York")
#print("================ Reston =============================")
await agent_tester.run_remote_agent_prompt(remote_agent, "What is the weather for Reston, VA")
#print("================ Los Angeles ========================")
await agent_tester.run_remote_agent_prompt(remote_agent, "What is the weather for Los Angeles, CA")

================ New York ===========================
📍 Weather for New York City, New York 📅 Friday, August 2, 2024                

🌤️ Conditions: Sunny then Chance Showers And Thunderstorms 🌡️ Temperature: High:
86°F | Low: 77°F 💧 Humidity: Not available 💨 Wind: Southwest wind 3 to 8 mph  
🌧️ Precipitation: 40% chance of showers and thunderstorms 👁️ Visibility: Not    
available 🌅 Sunrise: Not available 🌇 Sunset: Not available                    
📍 Weather for Reston, VA 📅 Friday, July 19, 2024                              

🌤️ Conditions:     Chance Showers And Thunderstorms then Mostly Cloudy 🌡️       
Temperature:    90°F  |  High: 90°  Low: 71° 💧 Humidity:       N/A 💨 Wind:    
Southwest wind 5 to 8 mph (afternoon) / Southwest wind 3 to 8 mph (tonight) 🌧️  
Precipitation:  70% chance of showers and thunderstorms 👁️ Visibility:     N/A  
🌅 Sunrise:        N/A 🌇 Sunset:         N/A                                   
📍 Weather for Los Angeles, CA 📅 Friday, June 7, 2024             

# Below is a screen shot showing the created agent deployment.

![](./challenge-5-agent-platform.png)